# Artea A/B Test: Coupon Targeting Strategy Analysis

**Background:** A**** conducted a randomized A/B experiment to test whether a 20% discount coupon increases purchases among recent website visitors who had not transacted in the last two months (~5,000 users). Half of the users were randomly assigned to receive a one-month, non-transferable coupon (treatment), while the remaining half formed the control group. The dataset includes post-experiment outcomes (transactions and net revenue), the treatment indicator, and pre-treatment customer characteristics covering acquisition channel, past purchase history, recent browsing behavior, and predicted demographic attributes.

**Goal:**
The main goal of this analysis is to measure the causal impact of the 20% discount coupon on purchasing behavior and revenue, and determine whether Artea should send the coupon to all customers or only to specific segments to maximize revenue.

**Roadmap:** In the following sections, we first (i) evaluate the success of the randomization and (ii) estimate the average treatment effect of the coupon on purchasing outcomes. Next, we (iii) investigate heterogeneous treatment effects to understand which customer segments respond more strongly to the discount, focusing on behavioral, historical, and demographic differences. We then (iv) develop and compare different targeting strategies by analyzing how sending coupons to everyone versus only selected customers affects revenue and profitability. Finally, based on the results, we (v) provide practical recommendations on how Artea can design more effective and profit-oriented future marketing campaigns.

# 0. Load data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.iolib.summary2 import summary_col
from IPython.display import display, Math, Latex, HTML

sns.set_theme(style='whitegrid')

In [ ]:
# Load data
data = pd.read_excel('IMA2026_WebsiteCaseData_with_Demog.xlsx', sheet_name="AB_test_demog")

print(f"Dataset shape: {data.shape}")
data.head()

We load the dataset `IMA2026_WebsiteCaseData_with_Demog.xlsx` (sheet `AB_test_demog`), which contains 5,000 user-level observations with no missing values across 12 variables.

Each row corresponds to a unique user and includes post-treatment outcomes (`trans_after`, `revenue_after`), the treatment indicator (`test_coupon`), demographic characteristics, acquisition channel, and several pre-treatment behavioral variables such as past purchases, recency, browsing activity, and cart abandonment.

In [ ]:
data.info()

In [ ]:
data.nunique()

In [ ]:
data['channel_acq'].unique()

In [ ]:
data['channel_acq'] = data['channel_acq'].replace({
    1: "Google", 2: "Facebook", 3: "Instagram", 4: "Referral", 5: "Other"
})

data['id'] = data['id'].astype(str)

In [ ]:
data['buy_before'] = (data['num_past_purch']>0).astype(int)

We recode `channel_acq` into meaningful category labels and construct a binary variable `buy_before` indicating whether the user had any prior purchases.

The dataset is clean, complete, and suitable for analysis.

# 1. Preliminary Checks

## 1.1. Randomization Check

**Simple balance check**

In [ ]:
from scipy.stats import ttest_ind

print("\nTREATMENT BALANCE CHECK")
print(f"Control group size: {(data['test_coupon']==0).sum()}")
print(f"Treatment group size: {(data['test_coupon']==1).sum()}")

balance_vars = ['num_past_purch', 'spent_last_purchase',
                'weeks_since_visit', 'browsing_minutes', 'shopping_cart',
                'minority', 'non_male']

rows = []
for var in balance_vars:
    ctrl = data.loc[data['test_coupon']==0, var]
    treat = data.loc[data['test_coupon']==1, var]
    t, p = ttest_ind(treat, ctrl)
    rows.append({'Variable': var, 'Control': ctrl.mean(),
                 'Treatment': treat.mean(),
                 'Diff': treat.mean()-ctrl.mean(), 'p-value': p})
pd.DataFrame(rows).set_index('Variable')

We first compare pre-treatment covariate means between treatment and control groups. No covariate difference is statistically significant at the 5% level, suggesting good balance and no obvious selection into treatment.

Balanced covariates support the assumption that differences in outcomes can be attributed to the coupon rather than pre-existing user differences.



**Check randomization by logit model**

In [ ]:
logit_formula = """test_coupon ~ C(channel_acq) + minority + non_male + num_past_purch
+ spent_last_purchase + weeks_since_visit + browsing_minutes + shopping_cart"""

logit_model = smf.logit(logit_formula, data=data).fit()
logit_model.summary()



We run a logistic regression with `test_coupon` as the dependent variable and the observed covariates as predictors.

- The logit model's Pseudo $R^2$ is near zero, confirming no systematic selection into treatment.

- Most coefficients are statistically insignificant, indicating that treatment assignment is not systematically explained by observed user characteristics.

This provides evidence consistent with random assignment, strengthening the causal interpretation of estimated treatment effects.

## 1.2. Covariate Structure

We examine covariate interrelationships to inform modeling choices in later sections. Low collinearity means interaction terms can be estimated cleanly

**Collinearity check**

In [ ]:
from statsmodels.iolib.summary2 import summary_col
from IPython.display import display, Math, Latex, HTML

fmla1 = """num_past_purch ~ C(channel_acq) + minority + non_male +
weeks_since_visit + browsing_minutes + shopping_cart"""
lm_wotest1 = smf.ols(formula=fmla1, data=data).fit()

fmla2 = """spent_last_purchase ~ C(channel_acq) + minority + non_male  +
weeks_since_visit + browsing_minutes + shopping_cart"""
lm_wotest2 = smf.ols(formula=fmla2, data=data).fit()

# Reporting
summary = summary_col(
    [lm_wotest1, lm_wotest2],
    stars=True,
    float_format="%.3f",
    model_names=['Num Past Purchases', 'Spent Past Purchases']
)

html_output = summary.as_html()
display(HTML(html_output))

We also check whether covariates are strongly related to each other and to baseline purchase behavior.

- Pre-treatment characteristics show minimal association with past purchase behavior ($R^2 < 0.01$), indicating a relatively homogeneous customer base.

- Only minority status showed a small negative association with past purchase frequency ($\beta = -0.210, p < 0.05$).

This homogeneity, combined with successful randomization, provides a strong foundation for unbiased treatment effect estimation

**Correlation check**

In [ ]:
corr = data[[
    "minority",
    "non_male",
    "num_past_purch",
    "shopping_cart",
    "browsing_minutes",
    "spent_last_purchase",
    "weeks_since_visit"
]].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Pairwise Correlation of Pre-Treatment Covariates", fontweight = 'bold')
plt.show()

A correlation heatmap confirms that most covariates are weakly correlated, with only a few moderate relationships. This indicates that the feature set captures distinct behavioral signals rather than redundant information.

Because the predictors are not highly collinear, interaction terms and treatment-effect heterogeneity analyses are less likely to be distorted by overlapping covariates.

## 1.3 Outcome Distribution

In [ ]:
data['test_coupon_yn'] = data['test_coupon'].map({1: 'Yes', 0:'No'})

fig, ax = plt.subplots(1,2, figsize = (12,6))

ax0 = ax[0]
sns.histplot(data, x = 'trans_after', hue = 'test_coupon_yn',
             multiple = 'dodge', ax = ax0)
legend = ax0.get_legend()
legend.set_title('Coupon')

ax1=ax[1]
sns.histplot(data, x = 'revenue_after', hue = 'test_coupon_yn',
             multiple = 'dodge', bins = 20, ax=ax1)
ax1.get_legend().remove()
plt.show()

Most users generate zero revenue, and spending among buyers is right-skewed.

**Section 1 Summary**

We verify the validity of the experiment and assess the covariate structure before proceeding to treatment effect estimation.

- The treatment and control groups are approximately equal in size and display very similar baseline characteristics.
  A logistic regression with `test_coupon` as the dependent variable shows no systematic determinants of assignment , confirming that A/B experiment is well randomized.

- Pre-treatment covariates are weakly correlated and show minimal collinearity. This means interaction terms in later models can be estimated without distortion from overlapping covariates.

Together, these checks support unbiased treatment effect estimation and reliable heterogeneity analysis in subsequent sections.

However, the outcome distributions reveal heavy zero-inflation and right skewness, which should be kept in mind for subsequent analysis.

# 2. Global effects Analysis

## 2.1. Simple t-test

In [ ]:
results = []

for metric in ['revenue_after', 'trans_after']:
    treatment = data[data['test_coupon'] == 1][metric]
    control = data[data['test_coupon'] == 0][metric]

    t_stat, p_value = stats.ttest_ind(treatment, control, equal_var=False)

    results.append({
        'Metric': metric,
        'Treatment Mean': treatment.mean(),
        'Control Mean': control.mean(),
        'Difference': treatment.mean() - control.mean(),
        'T-Statistic': t_stat,
        'p-value': p_value,
        'Significant (p<0.05)': 'yes' if p_value < 0.05 else 'no'
    })

results_df = pd.DataFrame(results).set_index('Metric')

results_df.style.format({'Treatment Mean': '{:.2f}',
                         'Control Mean': '{:.2f}',
                         'Difference': '{:.2f}',
                         'T-Statistic': '{:.3f}',
                         'p-value': '{:.4f}'})

A simple difference-in-means t-test shows no statistically significant effect of coupons on post-period revenue, but a small and significant increase in the number of transactions. This suggests the coupon may stimulate purchasing frequency without translating into higher total revenue.

Because the average revenue effect is not positive, blanket couponing is unlikely to be optimal; targeting should focus on segments where incremental revenue is positive.

## 2.2. Model-Based

**Revenue after**

In [ ]:
lm_rev = smf.ols("revenue_after ~ test_coupon", data=data).fit()
lm_rev2 = smf.ols("revenue_after ~ test_coupon + minority + non_male", data=data).fit()
lm_rev3 = smf.ols("revenue_after ~ test_coupon + minority + non_male + weeks_since_visit + num_past_purch \
+ spent_last_purchase + browsing_minutes + shopping_cart", data=data).fit()
lm_rev4 = smf.ols("revenue_after ~ test_coupon + minority + non_male + weeks_since_visit + num_past_purch \
+ spent_last_purchase + browsing_minutes + shopping_cart + C(channel_acq)", data=data).fit()

# Reporting
summary = summary_col(
    [lm_rev, lm_rev2, lm_rev3, lm_rev4],
    stars=True,
    float_format="%.3f",
    model_names=['REVENUE 1', 'REVENUE 2', 'REVENUE 3', 'REVENUE 4']
)

html_output = summary.as_html()
display(HTML(html_output))


Regression models with increasing sets of controls confirm the model-free result: the coupon coefficient remains small and statistically insignificant for revenue. In contrast, several pre-treatment covariates (e.g., shopping_cart, past purchases, recency) are strongly associated with revenue, improving explanatory power once included.

The lack of a positive average revenue effect reinforces the need to look for heterogeneous treatment effects rather than relying on a single global estimate.

**Transactions after**

In [ ]:
lm_trans = smf.ols("trans_after ~ test_coupon", data=data).fit()
lm_trans2 = smf.ols("trans_after ~ test_coupon + minority + non_male", data=data).fit()
lm_trans3 = smf.ols("trans_after ~ test_coupon + minority + non_male + weeks_since_visit + num_past_purch \
+ spent_last_purchase + browsing_minutes + shopping_cart", data=data).fit()
lm_trans4 = smf.ols("trans_after ~ test_coupon + minority + non_male + weeks_since_visit + num_past_purch \
+ spent_last_purchase + browsing_minutes + shopping_cart + C(channel_acq)", data=data).fit()

# Reporting
summary = summary_col(
    [lm_trans, lm_trans2, lm_trans3, lm_trans4],
    stars=True,
    float_format="%.3f",
    model_names=['TRANSACTIONS 1', 'TRANSACTIONS 2', 'TRANSACTIONS 3', 'TRANSACTIONS 4']
)

html_output = summary.as_html()
display(HTML(html_output))

For transactions, the coupon effect is positive and statistically significant across specifications, indicating that coupons increase the probability/number of purchases. However, this uplift in transactions does not automatically imply higher revenue, consistent with the revenue results above.

**Section 2 Summary**

Coupons significantly increase the number of transactions but do not generate a statistically significant lift in average revenue. This pattern (more transactions but no revenue lift) is consistent with cannibalization or lower basket size, and further supports targeting coupons to users with positive incremental revenue.

Given the weak average revenue effect but a positive transactions effect, the next step is to test whether coupon impact varies by user intent and history (heterogeneous treatment effects) and to design a targeted policy accordingly.

# 3. Heterogeneous Treatment Effects

## 3.1. Model free analysis

We perform a model-free heterogeneity analysis by splitting the sample into subgroups (moderators) and comparing treated vs control mean outcomes within each subgroup.

In [ ]:
def calculate_group_stats(df, group_column, metrics=['trans_after', 'revenue_after']):
  """Calculates t-test results and means for different groups of a column."""

  groups = df[group_column].unique()
  results = []

  for group_value in groups:
    result_row = {'group': group_value}

    for metric in metrics:
      group1 = df[(df['test_coupon'] == 1) & (df[group_column] == group_value)][metric]
      group2 = df[(df['test_coupon'] == 0) & (df[group_column] == group_value)][metric]
      t_statistic, p_value = stats.ttest_ind(group1, group2)
      mean_group1 = group1.mean()
      mean_group2 = group2.mean()
      overall_mean = df[df[group_column] == group_value][metric].mean()

      result_row[f'mean_{metric}_cp_1'] = mean_group1
      result_row[f'mean_{metric}_cp_0'] = mean_group2
      result_row[f'mean_diff_{metric}'] = mean_group1 - mean_group2
      result_row[f'overall_mean_{metric}'] = overall_mean
      result_row[f't_stat_{metric}'] = t_statistic
      result_row[f'p_value_{metric}'] = p_value

    results.append(result_row)

  return pd.DataFrame(results)

In [ ]:
moderators = ['minority', 'non_male', 'channel_acq', 'buy_before', 'shopping_cart']

# Collect results for all moderators
all_results = []
for var in moderators:
    df_res = calculate_group_stats(data, var, metrics=['trans_after', 'revenue_after'])
    df_res.insert(0, 'moderator', var)
    all_results.append(df_res)

combined = pd.concat(all_results, ignore_index=True)

def format_col(col):
    return col.replace('_', ' ').replace(' ', '\n', 0)

combined.columns = [format_col(c) for c in combined.columns]
combined.set_index(['moderator', 'group'], inplace=True)

In [ ]:
def highlight_significance(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for col in df.columns:
        if 'p value' in col:
            # Find corresponding t-statistic column
            metric_part = col.replace('p value', '')
            t_col = 't stat' + metric_part

            for idx in df.index:
                p = df.loc[idx, col]
                if p < 0.01:
                    styles.loc[idx, col] = 'background-color: darkgreen; color: white'
                    if t_col in df.columns:
                        styles.loc[idx, t_col] = 'background-color: darkgreen; color: white'
                elif p < 0.05:
                    styles.loc[idx, col] = 'background-color: mediumseagreen; color: white'
                    if t_col in df.columns:
                        styles.loc[idx, t_col] = 'background-color: mediumseagreen; color: white'
                elif p < 0.1:
                    styles.loc[idx, col] = 'background-color: palegreen; color: black'
                    if t_col in df.columns:
                        styles.loc[idx, t_col] = 'background-color: palegreen; color: black'
    return styles

combined.style.apply(highlight_significance, axis=None)


Key patterns emerge:
- Users with `shopping_cart`=1 show significantly higher coupon effects on transactions (p < 0.01), suggesting purchase intent amplifies coupon effectiveness.
- The coupon effect on transactions is significant for non-minority, non-male, and Facebook/Instagram-acquired users, but the revenue effects are mostly insignificant across subgroups.
- Google-acquired users show a significant negative revenue effect (p < 0.01), suggesting coupons may cannibalize revenue in this segment.

This highlights that coupon effects are not uniform across users and depend on behavioral and acquisition characteristics, motivating moving beyond the global average effect and designing a targeted coupon policy.

It is, however, noted that some subgroup cells can be small, so individual subgroup t-tests may be noisy, so we treat this as exploratory evidence and confirm patterns with model-based interaction terms.

In [ ]:
import matplotlib.patches as mpatches

data['spent_last_purchase_bin'] = pd.cut(
    data['spent_last_purchase'],
    bins=range(0, int(data['spent_last_purchase'].max()) + 20, 20),
    right=False
)

cp_yes_raw = data[data['test_coupon'] == 1]
cp_no_raw  = data[data['test_coupon'] == 0]

def get_grouped(df, var):
    return df.groupby(var, observed=True).agg(
        trans_after=('trans_after', 'mean'),
        revenue_after=('revenue_after', 'mean')
    )

discrete_vars = ['num_past_purch', 'browsing_minutes', 'weeks_since_visit', 'spent_last_purchase_bin']

plot_items = []

for var in discrete_vars:
    grp_yes = get_grouped(cp_yes_raw, var)
    grp_no  = get_grouped(cp_no_raw, var)
    common_idx = grp_yes.index.intersection(grp_no.index)
    grp_yes = grp_yes.loc[common_idx]
    grp_no  = grp_no.loc[common_idx]
    plot_items.append((grp_yes, grp_no, var, 'bar'))

BAR_COLORS  = {'yes': 'orange', 'no': 'royalblue'}
LINE_COLORS = {'yes': 'darkorange', 'no': 'blue'}

n_vars = len(plot_items)
n_cols = 2
n_rows = (n_vars + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()
width = 0.35

for i, (grp_yes, grp_no, title, style) in enumerate(plot_items):
    ax_bar  = axes[i]
    ax_line = ax_bar.twinx()

    if style == 'bar':
        x     = np.arange(len(grp_yes.index))
        ticks = [str(v) for v in grp_yes.index]

        ax_bar.bar(x - width/2, grp_yes['revenue_after'], width,
                   color=BAR_COLORS['yes'], alpha=0.45)
        ax_bar.bar(x + width/2, grp_no['revenue_after'],  width,
                   color=BAR_COLORS['no'],  alpha=0.45)
        ax_line.plot(x, grp_yes['trans_after'], color=LINE_COLORS['yes'],
                     marker='o', linewidth=1.8)
        ax_line.plot(x, grp_no['trans_after'],  color=LINE_COLORS['no'],
                     marker='o', linewidth=1.8)

        ax_bar.set_xticks(x)
        ax_bar.set_xticklabels(ticks, fontsize=8)
        ax_bar.set_xticklabels([t.replace(', ', ',\n') for t in ticks], fontsize=7)

    ax_bar.set_xlabel(title, fontsize=9)
    ax_bar.set_ylabel('Avg Revenue After (USD)', color='gray', fontsize=8)
    ax_line.set_ylabel('Avg Transactions After', color='gray', fontsize=8)
    ax_bar.set_title(f'By {title}', fontsize=10, fontweight='bold')
    ax_bar.grid(False)
    ax_line.grid(False)

for j in range(n_vars, len(axes)):
    axes[j].set_visible(False)

legend_elements = [
    mpatches.Patch(facecolor=BAR_COLORS['yes'], alpha=0.55, label='Coupon — Avg Revenue'),
    mpatches.Patch(facecolor=BAR_COLORS['no'],  alpha=0.55, label='No Coupon — Avg Revenue'),
    plt.Line2D([0],[0], color=LINE_COLORS['yes'], marker='o', lw=2, label='Coupon — Avg Transactions (line)'),
    plt.Line2D([0],[0], color=LINE_COLORS['no'],  marker='o', lw=2, label='No Coupon — Avg Transactions (line)'),
]

fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.04))

fig.suptitle('Revenue & Transactions by Customer Segment — Coupon vs No Coupon',
             fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

We visualize treated vs control outcomes across key behavioral segments (past purchases, browsing minutes, recency, prior spend). The plots illustrate that coupons increase transactions more consistently than revenue, while revenue responses vary substantially by segment.

This pattern is consistent with heterogeneous uplift: coupons can be beneficial for some high-intent or low-frequency segments but may be neutral or negative for others, reinforcing the need for targeting.

Some extreme segments contain very few users, so their treated–control mean differences can be unstable and driven by outliers. For targeting decisions, we therefore rely primarily on model-based interaction estimates, which use the full sample and provide more stable uplift rankings.

## 3.2. Model based analysis

We estimate OLS regressions with interaction terms of the form `test_coupon` × `moderator` to formally test which covariates significantly moderate the coupon effect.

For each moderator, we run both a simple specification (interaction only) and a full specification (with all controls).

In [ ]:
def run_interaction_table(data, interactions, feature_cols, browsing_cols,
                          outcome='buy_after', treatment='test_coupon', model='logit'):
    """
    Runs simple and full models for each interaction term and returns a styled summary table.

    Parameters
    ----------
    data          : pd.DataFrame
    interactions  : list of str  – variables to interact with treatment
    feature_cols  : list of str  – base feature controls
    browsing_cols : list of str  – additional browsing controls
    outcome       : str          – dependent variable (default: 'buy_after')
    treatment     : str          – treatment variable (default: 'test_coupon')
    model         : str          – model type: 'logit', 'ols' (default: 'logit')

    Returns
    -------
    styled : pandas Styler object
    """
    model_map = {
        'logit':  smf.logit,
        'ols':    smf.ols,
    }
    if model not in model_map:
        raise ValueError(f"model must be one of {list(model_map.keys())}, got '{model}'")

    fit_model = model_map[model]

    all_results = {}
    for var in interactions:
        fmla_simple = f"{outcome} ~ {treatment} * {var}"
        model_simple = fit_model(formula=fmla_simple, data=data).fit(disp=False)
        all_results[f'{var}_simple'] = {'coef': model_simple.params, 'pval': model_simple.pvalues}

        controls = [v for v in (feature_cols + browsing_cols) if v != var]
        fmla_full = f"{outcome} ~ {treatment} * {var} + {' + '.join(controls)}"
        model_full = fit_model(formula=fmla_full, data=data).fit(disp=False)
        all_results[f'{var}_full'] = {'coef': model_full.params, 'pval': model_full.pvalues}

    all_vars = set()
    for r in all_results.values():
        all_vars.update(r['coef'].index)

    rows = []
    for var in sorted(all_vars):
        row = {'Variable': var}
        for name, r in all_results.items():
            if var in r['coef'].index:
                coef = r['coef'][var]
                pval = r['pval'][var]
                stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
                row[name] = f"{coef:.3f}{stars}"
                row[f'{name}_pval'] = pval
            else:
                row[name] = ''
                row[f'{name}_pval'] = np.nan
        rows.append(row)

    df = pd.DataFrame(rows)

    display_cols = ['Variable'] + [f'{v}_{t}' for v in interactions for t in ['simple', 'full']]
    df_display = df[display_cols].set_index('Variable')
    df_display.columns = [
        c.replace('_simple', '\n(simple)').replace('_full', '\n(full)')
        for c in df_display.columns
    ]

    def highlight_sig(row):
        if ':' not in row.name:
            return ['' for _ in row]
        return [
            'background-color: tomato; color: white; font-weight: bold'    if '***' in str(v) else
            'background-color: orange; color: white; font-weight: bold' if '**'  in str(v) else
            'background-color: gold; font-weight: bold'                 if '*'   in str(v) else ''
            for v in row
        ]

    styled = df_display.style.apply(highlight_sig, axis=1)
    styled = styled.set_caption(
        f"Dependent variable: {outcome} | Model: {model.upper()}<br>*p<0.1; **p<0.05; ***p<0.01"
    )
    return styled

We split the covariates into two groups:
- core features (demographics, acquisition channel, purchase history, recency) and
- browsing behavior (browsing minutes, shopping cart) — to keep the tables readable.

In [ ]:
feature_cols = ['channel_acq', 'minority', 'non_male', 'num_past_purch', 'spent_last_purchase', 'weeks_since_visit']
browsing_cols = ['browsing_minutes', 'shopping_cart']

run_interaction_table(data, interactions=feature_cols, feature_cols=feature_cols, browsing_cols=browsing_cols,
                      outcome='revenue_after', treatment='test_coupon', model = 'ols')

In [ ]:
run_interaction_table(data, interactions=browsing_cols, feature_cols=feature_cols, browsing_cols=browsing_cols,
                      outcome='revenue_after', treatment='test_coupon', model = 'ols')

Results show that `test_coupon` × `shopping_cart` is positive and significant, indicating higher incremental revenue for high-intent users, while `test_coupon` × `num_past_purch` is negative and significant, indicating diminishing effectiveness among frequent buyers.

## 3.3. Decomposing the Coupon Effect


Before defining specific targeting rules, we further examine whether the coupon affects revenue through different channels: getting more customers to buy versus changing how much buyers spend.

The following plots illustrate that treatment effects are not homogeneous. Instead, they vary across acquisition channels, purchase intent signals, and customer loyalty levels and differ depending on whether we look at purchase incidence or spending conditional on purchase.

These patterns suggest that a single revenue model may not fully capture the structure of incremental revenue.

In [ ]:
def plot_means_ci(df, group_col, outcome_col, treat_col="test_coupon",
                  title=None, ax=None, order=None, dodge=0.08):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 3))
    else:
        fig = ax.figure

    stats_df = (
        df.groupby([group_col, treat_col], as_index=False, observed=True)
          .agg(mean=(outcome_col, "mean"),
               std=(outcome_col, "std"),
               n=(outcome_col, "count"))
    )
    stats_df["se"] = stats_df["std"] / np.sqrt(stats_df["n"])
    stats_df["lo"] = stats_df["mean"] - 1.96 * stats_df["se"]
    stats_df["hi"] = stats_df["mean"] + 1.96 * stats_df["se"]

    if order is None:
        order = df[group_col].dropna().unique().tolist()

    x = np.arange(len(order))
    base = pd.DataFrame({group_col: order, "x": x})

    colors = {0: "steelblue", 1: "darkorange"}

    for tval, label, offset, ls in [
        (0, "Control", -dodge, "-"),
        (1, "Coupon",   dodge, "--"),
    ]:
        s = stats_df[stats_df[treat_col] == tval][[group_col, "mean", "lo", "hi"]]
        aligned = base.merge(s, on=group_col, how="left").sort_values("x")
        x_pos = aligned["x"].to_numpy() + offset
        c = colors[tval]

        ax.plot(x_pos, aligned["mean"], marker="o", linestyle=ls,
                color=c, label=label, zorder=3)
        ax.vlines(x_pos, aligned["lo"], aligned["hi"], color=c, alpha=0.7, linewidth=1.5)
        ax.scatter(x_pos, aligned["lo"], marker="_", color=c, s=40, zorder=3)
        ax.scatter(x_pos, aligned["hi"], marker="_", color=c, s=40, zorder=3)

    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_ylabel(outcome_col, fontsize=9)
    ax.grid(True, axis="y", alpha=0.2)
    ax.set_xticks(x)
    ax.set_xticklabels(order, fontsize=8)
    ax.legend(fontsize=8)

    return fig, ax

In [ ]:
data["past_bin"] = pd.cut(
    data["num_past_purch"],
    bins=[-0.1, 0.5, 1.5, 2.5, np.inf],
    labels=["0", "1", "2", "3+"]
)

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(14, 8))

past_order = ["0", "1", "2", "3+"]

# Row 1: Revenue
plot_means_ci(data, "past_bin", "revenue_after",
              title="Past Purchases × Coupon (Revenue)", ax=axes[0, 0], order=past_order)
plot_means_ci(data, "channel_acq", "revenue_after",
              title="Channel × Coupon (Revenue)", ax=axes[0, 1])
plot_means_ci(data, "shopping_cart", "revenue_after",
              title="Shopping Cart × Coupon (Revenue)", ax=axes[0, 2])

# Row 2: Transactions
plot_means_ci(data, "past_bin", "trans_after",
              title="Past Purchases × Coupon (Transactions)", ax=axes[1, 0], order=past_order)
plot_means_ci(data, "channel_acq", "trans_after",
              title="Channel × Coupon (Transactions)", ax=axes[1, 1])
plot_means_ci(data, "shopping_cart", "trans_after",
              title="Shopping Cart × Coupon (Transactions)", ax=axes[1, 2])

plt.tight_layout()
plt.show()

The coupon lifts transactions across most segments, but revenue effects are weaker and mixed. This suggests the coupon may operate through two distinct channels: whether a customer purchases at all (extensive margin) and how much a buyer spends (intensive margin). We decompose below.

In [ ]:
data['buy_after'] = (data['trans_after'] > 0).astype(int)

In [ ]:
buyers = data[data["buy_after"] == 1].copy()
buyers["past_bin"] = pd.cut(
    buyers["num_past_purch"],
    bins=[-0.1, 0.5, 1.5, 2.5, np.inf],
    labels=["0", "1", "2", "3+"]
)

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(14, 8))

# Row 1: Extensive margin — P(buy) for ALL users
plot_means_ci(data, "past_bin", "buy_after",
              title="Past Purchases (P(Purchase))", ax=axes[0, 0], order=past_order)
plot_means_ci(data, "channel_acq", "buy_after",
              title="Channel (P(Purchase))", ax=axes[0, 1])
plot_means_ci(data, "shopping_cart", "buy_after",
              title="Shopping Cart (P(Purchase))", ax=axes[0, 2])

# Row 2: Intensive margin — Revenue among BUYERS ONLY
plot_means_ci(buyers, "past_bin", "revenue_after",
              title="Past Purchases (Revenue | Buy)", ax=axes[1, 0], order=past_order)
plot_means_ci(buyers, "channel_acq", "revenue_after",
              title="Channel (Revenue | Buy)", ax=axes[1, 1])
plot_means_ci(buyers, "shopping_cart", "revenue_after",
              title="Shopping Cart (Revenue | Buy)", ax=axes[1, 2])

for ax in axes[0, :]: ax.set_ylabel("P(Purchase)")
for ax in axes[1, :]: ax.set_ylabel("Revenue | Buy (USD)")

plt.tight_layout()
plt.show()

The top row confirms that the coupon increases purchase probability across moderators, with the gap between coupon and control widening for low-purchase and cart-abandoning users. The bottom row tells a different story: among buyers, coupon recipients consistently spend less than control buyers across all segments.

In [ ]:
buyers = data[data["trans_after"] > 0]

# Extensive margin
n_treat = (data["test_coupon"] == 1).sum()
n_ctrl  = (data["test_coupon"] == 0).sum()
buy_treat = data.loc[data["test_coupon"] == 1, "buy_after"].sum()
buy_ctrl  = data.loc[data["test_coupon"] == 0, "buy_after"].sum()
z_ext, p_ext = proportions_ztest([buy_treat, buy_ctrl], [n_treat, n_ctrl])

# Intensive margin
rev_treat = buyers.loc[buyers["test_coupon"] == 1, "revenue_after"]
rev_ctrl  = buyers.loc[buyers["test_coupon"] == 0, "revenue_after"]
t_int, p_int = stats.ttest_ind(rev_treat, rev_ctrl, equal_var=False)

results = pd.DataFrame({
    "Margin": ["Extensive (Purchase Rate)", "Intensive (Revenue | Buy)"],
    "Control": [f"{buy_ctrl/n_ctrl:.4f}", f"${rev_ctrl.mean():.2f}"],
    "Coupon":  [f"{buy_treat/n_treat:.4f}", f"${rev_treat.mean():.2f}"],
    "Difference": [f"{buy_treat/n_treat - buy_ctrl/n_ctrl:+.4f}",
                   f"${rev_treat.mean() - rev_ctrl.mean():+.2f}"],
    "Test Statistic": [f"z = {z_ext:.3f}", f"t = {t_int:.3f}"],
    "p-value": [round(p_ext, 4), round(p_int, 4)],
    "N": [f"{n_ctrl} / {n_treat}", f"{len(rev_ctrl)} / {len(rev_treat)}"]
}).set_index("Margin")

results

The statistical tests confirm the visual pattern. The coupon significantly increases purchase probability (extensive margin), but among buyers, coupon recipients spend significantly less (intensive margin).

These two opposing forces explain the weak overall revenue effect from Section 2. A single revenue model conflates two processes that respond to the coupon in opposite directions.

**Section 3 Summary**

The global effects analysis in from Section 2 suggests that while coupons stimulate purchasing activity, incremental revenue gains are largely offset by discounting, making a blanket coupon strategy suboptimal.

To address this, we analyze heterogeneous treatment effects using interaction terms of the form `test_coupon` × moderator.

The heterogeneous treatment analysis shows that coupon effects vary substantially across customer segments. While the average revenue effect is weak overall, interaction results reveal a strongly positive and statistically significant effect within a specific target group (high-intent, low-loyalty users), and a negative or weak effect for non-target customers. This indicates that incremental profitability is driven by a specific subgroup of customers, while distributing coupons broadly across all users can offset gains and potentially reduce overall revenue.

The results from Section 3 support moving from a blanket treat-all strategy to a targeted coupon policy that concentrates discounts on users with positive predicted incremental revenue.

Taken together, these findings indicate that treatment effects vary across customers in a structured but potentially non-linear manner.

While simple heuristics may capture part of this heterogeneity, a more flexible modelling approach can better estimate customer-level incremental revenue.

We therefore evaluate three increasingly sophisticated targeting strategies:
1. A simple heuristic rule,
2. A linear interaction-based rule,
3. A flexible two-phase uplift model.

# 4. Targeting Rule Options


## Option 1 - Simple Heuristic Rule

The heterogeneous treatment effect analysis showed that coupon effectiveness is concentrated among customers with **cart abandonment** and **few prior purchases**.

This motivates a simple targeting rule:

> **Target if `num_past_purch ≤ threshold` AND `shopping_cart == 1`**

No statistical model is required — the threshold is set empirically based on observed treatment effects across cells.
The 2D grid below maps average treatment effects across `num_past_purch` × `shopping_cart` cells (minimum 20 observations per arm). This helps visually confirm the threshold for the heuristic rule.

In [ ]:
# Build heatmap_data with p-values

def compute_cell_stats(group, include_groups=False):
    treated = group[group['test_coupon'] == 1]['revenue_after']
    control = group[group['test_coupon'] == 0]['revenue_after']
    te      = treated.mean() - control.mean()
    n1, n0  = len(treated), len(control)
    if n1 > 1 and n0 > 1:
        _, pval = stats.ttest_ind(treated, control, equal_var=False)
    else:
        pval = np.nan
    return pd.Series({'te_revenue': te, 'n1': n1, 'n0': n0, 'pval': pval})

heatmap_data = (data.groupby(['shopping_cart', 'num_past_purch'])
                    .apply(compute_cell_stats, include_groups=False)
                    .reset_index())

# Filter unreliable cells
n_reliable = 10
heatmap_data = heatmap_data[(heatmap_data['n1'] >= n_reliable) & (heatmap_data['n0'] >= n_reliable)]

In [ ]:
# Pivot
pivot    = heatmap_data.pivot(index='shopping_cart', columns='num_past_purch', values='te_revenue')
pivot_n1 = heatmap_data.pivot(index='shopping_cart', columns='num_past_purch', values='n1')
pivot_n0 = heatmap_data.pivot(index='shopping_cart', columns='num_past_purch', values='n0')
pivot_p  = heatmap_data.pivot(index='shopping_cart', columns='num_past_purch', values='pval')

In [ ]:
# Derive threshold from cart abandoners
cart_data = (heatmap_data[heatmap_data['shopping_cart'] == 1]
             .sort_values('num_past_purch')
             .set_index('num_past_purch')['te_revenue'])

purch_threshold = None
for purch, te in cart_data.items():
    if te > 0:
        purch_threshold = purch
    else:
        break

if purch_threshold is None:
    print("→ Data-derived threshold: no positive consecutive cells from 0 upward.")
else:
    print(f"→ Data-derived threshold: num_past_purch <= {purch_threshold}")

In [ ]:
from matplotlib.patches import Rectangle

# Plot
def pval_stars(p):
    if np.isnan(p): return ''
    if p < 0.01:    return '***'
    if p < 0.05:    return '**'
    if p < 0.1:     return '*'
    return ''

fig, ax = plt.subplots(figsize=(12, 4))

vals = pivot.values
vmax = np.nanmax(np.abs(vals))
im = ax.imshow(vals, cmap='RdYlGn', vmin=-vmax, vmax=vmax, aspect='auto')

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(['No Cart (0)', 'Cart Abandoner (1)'])
ax.set_xlabel('num_past_purch')
ax.set_ylabel('shopping_cart')

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val  = pivot.values[i, j]
        n1   = pivot_n1.values[i, j]
        n0   = pivot_n0.values[i, j]
        pval = pivot_p.values[i, j]
        if not np.isnan(val):
            stars    = pval_stars(pval)
            pval_str = f'p={pval:.3f}{stars}' if not np.isnan(pval) else 'p=n/a'
            ax.text(j, i,
                    f'{val:.1f}{stars}\n(n1={int(n1)},\n n0={int(n0)})\n{pval_str}',
                    ha='center', va='center', fontsize=7.5,
                    color='black' if abs(val) < vmax * 0.6 else 'white')

# Target zone rectangle
if purch_threshold is not None:
    target_rows = [i for i, idx in enumerate(pivot.index)   if idx == 1]
    target_cols = [j for j, col in enumerate(pivot.columns) if col <= purch_threshold]

    if target_rows and target_cols:
        row_start = min(target_rows) - 0.5
        col_start = min(target_cols) - 0.5
        row_span  = len(target_rows)
        col_span  = len(target_cols)

        ax.add_patch(Rectangle((col_start, row_start), col_span, row_span,
                               linewidth=2.5, edgecolor='black', facecolor='none',
                               linestyle='--', label='Target zone (rule)'))

plt.colorbar(im, ax=ax, label='Treatment Effect on Revenue ($)')
ax.set_title(f'Coupon Treatment Effect on Revenue\n'
             f'by Past Purchases × Shopping Cart Status\n'
             f'(cells with n_treatment < {n_reliable} or n_control < {n_reliable} excluded)',
             fontweight='bold')
ax.legend(loc='upper right')
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
def option_simple_rule(data, purch_threshold):
    print("Simple heuristic rule")
    print(f"Rule: num_past_purch <= {purch_threshold} AND shopping_cart == 1")

    data['target_rule'] = (
        (data['num_past_purch'] <= purch_threshold) &
        (data['shopping_cart'] == 1)
    ).astype(int)

    print(f"\nTargeted: {data['target_rule'].sum()} / {len(data)} customers")

    return data

data = option_simple_rule(data, purch_threshold)

data[['id', 'target_rule']].head()

Running the rule at a threshold of `num_past_purch ≤ 2` and `shopping_cart = 1` flags a subset of customers for targeting and adds a binary `target_rule` column to the dataset.

In [ ]:
target_mask = (data["num_past_purch"] <= 2) & (data["shopping_cart"] == 1)
target_group = data[target_mask]

from scipy import stats
treated = target_group[target_group["test_coupon"] == 1]["revenue_after"]
control = target_group[target_group["test_coupon"] == 0]["revenue_after"]
t, p = stats.ttest_ind(treated, control, equal_var=False)
ATE = treated.mean() - control.mean()
print(f"Target zone ATE: {ATE:.2f}")
print(f"t={t:.3f}, p={p:.3f}, n_treated={len(treated)}, n_control={len(control)}")

The heatmap shows that positive treatment effects are concentrated among customers with `shopping_cart = 1` and `num_past_purch ≤ 2`, while effects weaken or turn negative for frequent buyers. The threshold `num_past_purch ≤ 2` is derived directly from consecutive positive cells among cart abandoners, ensuring a data-driven rule.

Within the target segment, the Average Treatment Effect is +3.83 USD (p = 0.007), statistically significant and economically meaningful, with more than 500 observations per arm. The implied total effect for targeted customers from the interaction model is approximately +4 USD, closely matching the segment-level estimate. Together, these findings confirm substantial treatment heterogeneity: the coupon increases revenue for high-intent, low-loyalty users but reduces revenue among frequent buyers.

In [ ]:
data["in_target"] = ((data["num_past_purch"] <= 2) & (data["shopping_cart"] == 1)).astype(int)

formula_robust = "revenue_after ~ test_coupon * in_target + minority + non_male + C(channel_acq) + weeks_since_visit + browsing_minutes + spent_last_purchase"
model_robust = smf.ols(formula_robust, data=data).fit(cov_type="HC3")
print(model_robust.summary())

As a robustness check, we estimate an OLS regression of `revenue_after` on `test_coupon`, `in_target` (the heuristic rule indicator), their interaction, and all covariates, using HC3 robust standard errors. The interaction term `test_coupon × in_target` equals +5.44 USD and is highly significant (p < 0.001), confirming that the targeting advantage is not driven by omitted variables. In contrast, the main effect of `test_coupon` is −1.40 USD (p = 0.061), indicating a slight revenue loss among non-target customers, consistent with cannibalization.


Overall, the heuristic rule isolates a profitable subgroup and improves performance relative to a global treat-all strategy. While simple and transparent, it remains a coarse segmentation approach that could be further refined with more flexible predictive targeting.


## Option 2 - OLS Model with Interactions

An OLS regression including interaction terms between `test_coupon` and the two key moderators captures heterogeneity while controlling for all covariates:

**Model Specification**

$$
\begin{aligned}
\text{revenue\_after} &= \beta_0
+ \beta_1 \cdot \text{test\_coupon} \\
&+ \beta_2 \cdot \text{num\_past\_purch}
+ \beta_3 \cdot (\text{test\_coupon} \times \text{num\_past\_purch}) \\
&+ \beta_4 \cdot \text{shopping\_cart}
+ \beta_5 \cdot (\text{test\_coupon} \times \text{shopping\_cart}) \\
&+ \dots
\end{aligned}
$$


**Individual Uplift Estimation**

$$
\hat{u}(x) =
\hat{y}(\text{test\_coupon}=1, X)
-
\hat{y}(\text{test\_coupon}=0, X)
$$

We target only customers for whom the model predicts a positive incremental effect from receiving the coupon, i.e. customers with a positive uplift  ($
\hat{u}(x) > 0
$) are targeted (`target_ols` = 1).

In [ ]:
COVARIATES = [
    'minority', 'non_male', 'C(channel_acq)',
    'num_past_purch', 'spent_last_purchase',
    'weeks_since_visit', 'browsing_minutes', 'shopping_cart'
]
COV_STR = ' + '.join(COVARIATES)

In [ ]:
def option_ols(data):
    print("OPTION 2: OLS with interaction terms")

    formula = (
        f"revenue_after ~ test_coupon + {COV_STR}"
        f" + test_coupon:num_past_purch"
        f" + test_coupon:shopping_cart"
    )

    model = smf.ols(formula=formula, data=data).fit(cov_type="HC3")
    print(model.summary())

    # predict uplift: score each person under coupon=1 and coupon=0
    dat_t1 = data.copy()
    dat_t1['test_coupon'] = 1
    dat_t0 = data.copy()
    dat_t0['test_coupon'] = 0

    data['uplift_ols'] = model.predict(dat_t1) - model.predict(dat_t0)
    data['target_ols'] = (data['uplift_ols'] > 0).astype(int)

    return data, model


In [ ]:
dat_ols, model_ols = option_ols(data)

In [ ]:
print(f"Targeted:     {dat_ols['target_ols'].sum()} / {len(dat_ols)} customers")
print(f"Mean uplift (targeted): ${dat_ols.loc[data['target_ols']==1, 'uplift_ols'].mean():.2f}")
print(f"Mean uplift (all):      ${dat_ols['uplift_ols'].mean():.2f}")

In [ ]:
import matplotlib.gridspec as gridspec
from statsmodels.nonparametric.smoothers_lowess import lowess

# Extract residuals & fitted values
fitted    = model_ols.fittedvalues
residuals = model_ols.resid
std_resid = residuals / residuals.std()

fig = plt.figure(figsize=(14, 8))
fig.suptitle("Model 2 — OLS Regression Diagnostics", fontsize=14, fontweight="bold")
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# Residuals vs. Fitted
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(fitted, residuals, alpha=0.25, s=12, color="steelblue", edgecolors="none")
ax1.axhline(0, color="crimson", linewidth=1.2, linestyle="--")
smooth = lowess(residuals, fitted, frac=0.3)
ax1.plot(smooth[:, 0], smooth[:, 1], color="darkorange", linewidth=1.8, label="LOWESS")
ax1.set_xlabel("Fitted values")
ax1.set_ylabel("Residuals")
ax1.set_title("1. Residuals vs. Fitted")
ax1.legend(fontsize=8)

# Q-Q Plot
ax2 = fig.add_subplot(gs[0, 1])
(osm, osr), (slope, intercept, r) = stats.probplot(std_resid, dist="norm")
ax2.scatter(osm, osr, alpha=0.25, s=12, color="steelblue", edgecolors="none")
ax2.plot(osm, slope * np.array(osm) + intercept, color="crimson", linewidth=1.5, linestyle="--")
ax2.set_xlabel("Theoretical quantiles")
ax2.set_ylabel("Standardised residuals")
ax2.set_title("2. Q-Q Plot of Residuals")

# Residuals vs. num_past_purch (continuous interaction)
ax3 = fig.add_subplot(gs[1, 0])
ax3.scatter(data["num_past_purch"], residuals, alpha=0.25, s=12, color="steelblue", edgecolors="none")
ax3.axhline(0, color="crimson", linewidth=1.2, linestyle="--")

group_means = pd.DataFrame({"x": data["num_past_purch"], "r": residuals}).groupby("x")["r"].mean()
ax3.plot(group_means.index, group_means.values, color="darkorange", linewidth=1.8, marker="o", markersize=4, label="Group mean")

ax3.set_xlabel("num_past_purch")
ax3.set_ylabel("Residuals")
ax3.set_title("3. Residuals vs. num_past_purch")
ax3.legend(fontsize=8)

# Residuals by shopping_cart (binary interaction)
ax4 = fig.add_subplot(gs[1, 1])
groups = [residuals[data["shopping_cart"] == g].values for g in [0, 1]]
bp = ax4.boxplot(groups, patch_artist=True, widths=0.4,
                 medianprops=dict(color="crimson", linewidth=2))
for patch, color in zip(bp["boxes"], ["#AED6F1", "#A9DFBF"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax4.axhline(0, color="crimson", linewidth=1.2, linestyle="--")
ax4.set_xticks([1, 2])
ax4.set_xticklabels(["No cart (0)", "Has cart (1)"])
ax4.set_ylabel("Residuals")
ax4.set_title("4. Residuals by shopping_cart")

# Annotate variance per group
for i, g in enumerate([0, 1]):
    var = residuals[data["shopping_cart"] == g].var()
    ax4.text(i + 1, ax4.get_ylim()[1] * 0.92, f"Var={var:.0f}",
             ha="center", fontsize=8, color="dimgray")

plt.show()

The model explains approximately 17.7% of the variation in post-treatment revenue (R² = 0.177; Adj. R² = 0.175), and the joint significance test strongly rejects the null that all coefficients are zero (F = 27.38, p < 0.001).

**Main Effects:**

- The average coupon effect (β = 1.204, p = 0.139) is not statistically significant. However, because interaction terms are included, this coefficient reflects the effect only when moderators equal zero and should not be interpreted as the overall treatment effect.

- Revenue increases strongly with prior purchase frequency (β = 3.788, p < 0.001) and with having an active shopping cart (β = 8.104, p < 0.001). Recency negatively predicts revenue (β = −1.116, p < 0.001), consistent with declining engagement over time.

**Interaction Effects:**

- The interaction between the coupon and prior purchase frequency is negative and statistically significant (β = −1.149, p = 0.009). This implies that the incremental impact of the coupon declines as customers become more experienced. In other words, coupons are more effective among less established customers and generate diminishing returns for highly loyal ones.

- The interaction between the coupon and having a shopping cart is positive and marginally significant (β = 2.998, p = 0.063). This suggests that coupons are particularly effective for customers who are already close to purchasing, acting as a conversion trigger.

Together, these findings indicate meaningful heterogeneity in treatment effects: the coupon is not universally beneficial but works best for specific customer segments.

**Targeting Implications:**

- Using predicted uplift, 3,119 out of 5,000 customers are selected for targeting. The mean predicted uplift among targeted customers is +1.61, while the average uplift across the full population is -0.28.

- This contrast highlights the importance of modeling heterogeneity. A uniform distribution of coupons across all customers would reduce revenue on average, whereas targeting based on predicted uplift generates positive incremental value.

**Overall Interpretation:**

The OLS model with interactions shows that coupon effectiveness is heterogeneous across customers, rather than constant for all individuals.

*   Although the average treatment effect is not statistically significant, the interaction terms reveal meaningful differences in response across customer segments.
*   The coupon is more effective for customers with fewer prior purchases and for those who already have items in their shopping cart, but less effective for highly experienced customers.
*   A uniform coupon strategy would reduce revenue on average, as overall uplift is negative. However, targeting customers based on predicted individual uplift generates positive incremental value.

Overall, modeling interaction effects enables a more profitable, data-driven targeting strategy by uncovering treatment heterogeneity that would be hidden in average effects. Targeting based on behavioral signals (e.g., past purchases, shopping cart activity, and recency) appears more effective than relying on broad demographic characteristics or a uniform coupon distribution.

## Option 3 - Two-Part Model

A two-part model treats this as two separate processes:

- **Part A**: Purchase incidence

  We estimate the probability of buying:
  $
\hat p_{tc}(x) \;=\; \Pr(\text{buy}=1 \mid T, X=x)
$ using a logistic regression.
- **Part B**: Spend conditional on buying

  Among buyers only $(\text{buy}=1)$, we model expected spend:
  $
\hat m_{tc}(x) \;=\; E\!\left[\text{revenue} \mid \text{buy}=1,\; T,\; X=x\right]
$ using a Gamma GLM with log link (fit on the subset with `revenue_after > 0`)


Individual-level uplift is the difference in expected revenue between the coupon ($t{=}1$) and no-coupon ($t{=}0$) scenarios:

$$\hat{u}(x) \;=\; \underbrace{\hat{p}_1(x)\;\hat{m}_1(x)}_{\text{expected revenue with coupon}} \;-\; \underbrace{\hat{p}_0(x)\;\hat{m}_0(x)}_{\text{expected revenue without coupon}}$$

A customer is **targeted** whenever $\hat{u}(x) > 0$, i.e. the coupon is predicted to increase their expected revenue.

**Part A — Screening interactions.** We first run simple logit models (one moderator × `test_coupon` at a time) to identify which variables significantly modify the coupon's effect on purchase incidence.

In [ ]:
feature_cols = ['channel_acq', 'minority', 'non_male', 'num_past_purch', 'spent_last_purchase', 'weeks_since_visit']
browsing_cols = ['browsing_minutes', 'shopping_cart']

# reusing run_interaction_table defined in Section 3
run_interaction_table(data, interactions=feature_cols, feature_cols=feature_cols, browsing_cols=browsing_cols,
                      outcome='buy_after', treatment='test_coupon', model = 'logit')

In [ ]:
run_interaction_table(data, interactions=browsing_cols, feature_cols=feature_cols, browsing_cols=browsing_cols,
                      outcome='buy_after', treatment='test_coupon', model = 'logit')

In [ ]:
# Part A: Purchase incidence model
formula_partA1 = """buy_after ~ test_coupon * num_past_purch
                            + test_coupon * shopping_cart
                            + channel_acq
                            + minority + non_male
                            + spent_last_purchase + weeks_since_visit
                            + browsing_minutes"""

formula_partA2 = """buy_after ~ test_coupon * num_past_purch
                            + shopping_cart
                            + channel_acq
                            + minority + non_male
                            + spent_last_purchase + weeks_since_visit
                            + browsing_minutes"""

formula_partA3 = """buy_after ~ test_coupon + num_past_purch
                            + shopping_cart
                            + channel_acq
                            + minority + non_male
                            + spent_last_purchase + weeks_since_visit
                            + browsing_minutes"""


Three Part A model variants are tested, differing in which interaction terms are included. Comparing their summaries guides selection of a parsimonious, well-specified model.

In [ ]:
model_A1 = smf.logit(formula_partA1, data=data).fit(disp=False)
model_A2 = smf.logit(formula_partA2, data=data).fit(disp=False)
model_A3 = smf.logit(formula_partA3, data=data).fit(disp=False)

for name, model in [("A1", model_A1), ("A2", model_A2), ("A3", model_A3)]:
    print(f"{name} — AIC: {model.aic:.2f} | BIC: {model.bic:.2f}")

In [ ]:
model_partA = model_A2

`model_A2` is selected as the final Part A model. It retains the `test_coupon × num_past_purch` interaction, confirming that the coupon's positive effect on purchase probability diminishes as prior purchase count increases.

In [ ]:
model_partA.summary()

**Results:**

* Model selection via AIC comparison favors specification A2, which retains the interaction between test_coupon and num_past_purch. The model is jointly significant (LLR p < 0.001) with a pseudo R² of 0.209, indicating meaningful explanatory power for purchase incidence.
* The coupon has a positive and statistically significant main effect (β = 0.502, p < 0.001), implying that it increases the likelihood of purchase. However, the interaction with prior purchase frequency is negative and significant (β = −0.089, p = 0.006), indicating diminishing marginal effectiveness: the coupon’s impact on purchase probability decreases as customers accumulate more prior purchases. This pattern is consistent with reduced price sensitivity among more experienced customers.


* Shopping cart status strongly predicts purchase (β = 1.460, p < 0.001),  reflecting purchase readiness, while recency negatively predicts buying (β = −0.186, p < 0.001). Demographic interactions are not retained, reinforcing the conclusion that behavioral rather than demographic variables drive heterogeneity in treatment response.

Overall, Part A shows that the coupon primarily operates by increasing the probability of purchase, particularly among less established customers.

**Part B — Conditional spend model.**


A Gamma GLM with a log link is appropriate for strictly positive, right-skewed spend data.

Forward AIC stepwise selection identifies the most predictive covariates for spend among buyers. The selected model is stored as `model_partB`.

In [ ]:
import itertools
import statsmodels.api as sm

candidates = ['num_past_purch', 'shopping_cart', 'spent_last_purchase',
    'weeks_since_visit', 'browsing_minutes', 'minority', 'non_male', 'C(channel_acq)']

def fit_gamma(formula, data):
    return smf.glm(
        formula,
        data=data,
        family=sm.families.Gamma(link=sm.families.links.Log())
    ).fit(cov_type="HC3")

buyers = data[data['buy_after'] == 1]
base_formula = "revenue_after ~ test_coupon"
current_vars, remaining = [], candidates.copy()
current_aic = fit_gamma(base_formula, buyers).aic

print(f"Forward Stepwise (AIC)  [n={len(buyers)}]")
print(f"  {'Step':<4} {'Added':<25} {'AIC':>8}")
print(f"  {'Base':<4} {'(test_coupon only)':<25} {current_aic:>8.1f}")

while remaining:
    best_aic, best_var = current_aic, None
    for var in remaining:
        m = fit_gamma(base_formula + " + " + " + ".join(current_vars + [var]), buyers)
        if m and m.aic+1 < best_aic:
            best_aic, best_var = m.aic, var
    if not best_var:
        print(f"  {'—':<4} {'No improvement, stopping':<25}")
        break
    current_vars.append(best_var); remaining.remove(best_var)
    current_aic = best_aic
    print(f"  {len(current_vars):<4} {best_var:<25} {best_aic:>8.1f}")

best_formula = base_formula + " + " + " + ".join(current_vars)

print(f"\n Interaction Test (test_coupon:X)")
print(f"  {'Variable':<25} {'AIC':>8}  {'':>4}")
for var in current_vars:
    if 'C(' in var: continue
    m = fit_gamma(best_formula + f" + test_coupon:{var}", buyers)
    if m:
        better = m.aic < current_aic
        print(f"  {f'test_coupon:{var}':<25} {m.aic:>8.1f}")
        if better:
            current_aic, best_formula = m.aic, best_formula + f" + test_coupon:{var}"

model_partB = fit_gamma(best_formula, buyers)
print(f"\n Final: {best_formula}")
print(f"   AIC = {current_aic:.1f}")

In [ ]:
model_partB.summary()

In [ ]:
dat_T1 = data.copy()
dat_T1['test_coupon'] = 1

dat_T0 = data.copy()
dat_T0['test_coupon'] = 0

pA_T1 = model_A2.predict(dat_T1)
pA_T0 = model_A2.predict(dat_T0)

# Part B: predicted revenue under T=1 and T=0
rev_T1 = model_partB.predict(dat_T1)
rev_T0 = model_partB.predict(dat_T0)


data['uplift_twopart'] = pA_T1 * rev_T1 - pA_T0 * rev_T0

print(data['uplift_twopart'].describe())
print(f"Share targeted: {(data['uplift_twopart'] > 0).mean():.1%}")
data['target_twopart'] = (data['uplift_twopart'] > 0).astype(int)

**Results:**

*   The main coupon effect is negative and significant (β = −0.241, p < 0.001), indicating that among buyers, treated customers spend less on average—consistent with the mechanical discount effect of the coupon. However, the interaction between test_coupon and shopping_cart is positive and statistically significant (β = 0.129, p = 0.047), suggesting that for customers already close to purchase, the coupon partially offsets this reduction in conditional spend.

*   Prior purchase frequency positively predicts spending (β = 0.037, p < 0.001), while recency reduces conditional spend (β = −0.018, p = 0.013). Model diagnostics indicate a reasonable fit for buyer-level spending (Pseudo R² ≈ 0.184).

Overall, Part B shows that although the coupon reduces average spending among buyers, its effect varies across customers, particularly those with active shopping carts.

Individual uplift scores combine Part A and Part B predictions. Customers with `uplift_twopart > 0` are flagged as targets (`target_twopart = 1`).

In [ ]:
from matplotlib.gridspec import GridSpec
from sklearn.metrics import roc_curve, auc

fig = plt.figure(figsize=(14, 8))
gs  = GridSpec(2, 6, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle("Model 3 — Two-Part Model Diagnostics", fontsize=14, fontweight="bold")

axes = {
    "roc"         : fig.add_subplot(gs[0, 0:2]),
    "calib"       : fig.add_subplot(gs[0, 2:4]),
    "pred_actual" : fig.add_subplot(gs[0, 4:6]),
    "dist"        : fig.add_subplot(gs[1, 0:3]),
    "uplift"      : fig.add_subplot(gs[1, 3:6]),
}

buyers = data[data["buy_after"] == 1].copy()

# PART A

# 1. ROC Curve
ax = axes["roc"]
y_true  = data["buy_after"]
y_score = model_partA.predict(data)
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color="steelblue", linewidth=2, label=f"AUC = {roc_auc:.3f}")
ax.plot([0, 1], [0, 1], color="crimson", linestyle="--", linewidth=1.2, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Part A — ROC Curve (buy_after)")
ax.legend(fontsize=9)
ax.set_aspect("equal")

# 2. Calibration Plot
ax = axes["calib"]
pred_prob = model_partA.predict(data)
n_bins    = 10
bins      = np.percentile(pred_prob, np.linspace(0, 100, n_bins + 1))
bin_idx   = np.digitize(pred_prob, bins[1:-1])
bin_pred, bin_actual = [], []
for b in range(n_bins):
    mask = bin_idx == b
    if mask.sum() > 0:
        bin_pred.append(pred_prob[mask].mean())
        bin_actual.append(y_true[mask].mean())
ax.scatter(bin_pred, bin_actual, s=60, color="steelblue", zorder=3)
ax.plot(bin_pred, bin_actual, color="steelblue", linewidth=1.5)
ax.plot([0, 1], [0, 1], color="crimson", linestyle="--", linewidth=1.2, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Actual purchase rate")
ax.set_title("Part A — Calibration Plot")
ax.legend(fontsize=9)
ax.set_aspect("equal")

# PART B

# 3. Predicted vs. Actual Spend
ax = axes["pred_actual"]
actual_spend    = buyers["revenue_after"].values
predicted_spend = model_partB.fittedvalues.values
ax.scatter(actual_spend, predicted_spend, alpha=0.35, s=14, color="steelblue", edgecolors="none")
lim = max(actual_spend.max(), predicted_spend.max()) * 1.05
ax.plot([0, lim], [0, lim], color="crimson", linestyle="--", linewidth=1.2, label="Perfect fit")
ax.set_xlabel("Actual spend (buyers)")
ax.set_ylabel("Predicted spend")
ax.set_title("Part B — Predicted vs. Actual Spend")
ax.legend(fontsize=9)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect("equal")

# COMBINED

# 4. Spend Distribution: Actual vs. Predicted
ax = axes["dist"]
ax.hist(actual_spend,    bins=30, density=True, alpha=0.4, color="steelblue",  label="Actual")
ax.hist(predicted_spend, bins=30, density=True, alpha=0.4, color="darkorange", label="Predicted")
ax.set_xlabel("Revenue (buyers only)")
ax.set_ylabel("Density")
ax.set_title("Part B — Spend Distribution: Actual vs. Predicted")
ax.legend(fontsize=9)

# 5. Uplift Distribution by Buyer Status
ax = axes["uplift"]
for label, mask, color in [
    ("Non-buyer", data["buy_after"] == 0, "steelblue"),
    ("Buyer",     data["buy_after"] == 1, "darkorange")
]:
    ax.hist(data.loc[mask, "uplift_twopart"], bins=40, density=True,
            alpha=0.4, color=color, label=label)
ax.axvline(0, color="crimson", linestyle="--", linewidth=1.2, label="Uplift = 0")
ax.set_xlabel("Predicted uplift (two-part)")

ax.set_title("Combined — Uplift Distribution by Buyer Status")
ax.legend(fontsize=9)

fig.subplots_adjust(top=0.90)
plt.show()

**Results:**

*  The purchase model (Part A) shows strong predictive performance, with an AUC of 0.817, indicating good discrimination between buyers and non-buyers. The calibration plot suggests close alignment between predicted and observed purchase probabilities.
*  The coupon significantly increases the probability of purchase, but its effect diminishes as prior purchase frequency increases, confirming heterogeneous treatment effects on the conversion margin.

*  The conditional spend model (Part B), estimated using a Gamma GLM, appropriately captures the right-skewed distribution of buyer revenue. While the coupon reduces average conditional spending among buyers (consistent with a discount effect), its interaction with shopping cart status partially offsets this reduction.
*  Combining both components, the distribution of predicted uplift reveals substantial heterogeneity in treatment effects. Approximately 73.3% of customers exhibit positive expected incremental revenue under the two-part model.

Overall, the two-part framework shows that the coupon mainly drives revenue by increasing the likelihood of purchase, while its impact on spending among buyers is more complex and depends on customer characteristics.

## Evaluation

**Policy Evaluation: Inverse Propensity Weighting (IPW)**

*Objective*: Estimate the expected total revenue under different targeting policies in order to identify the strategy that maximizes Artea's revenue from the coupon campaign.

*Method*:

- Since coupons were assigned randomly in the experiment rather than according to our targeting rule, we use Inverse Propensity Weighting (IPW) to obtain an unbiased estimate of revenue under any given policy.

- For a binary treatment with equal assignment probability $e = P(W_i = 1) = 0.5$, the IPW estimator for total revenue under policy $\pi$ is:

$$\hat{R}(\pi) = \sum_{i=1}^{N} \left[ \frac{\mathbf{1}(\pi_i = 1) \cdot \mathbf{1}(W_i = 1)}{e} \cdot Y_i + \frac{\mathbf{1}(\pi_i = 0) \cdot \mathbf{1}(W_i = 0)}{1 - e} \cdot Y_i \right]$$

where:

- $\pi_i \in \{0, 1\}$: whether our policy targets customer i
- $W_i \in \{0, 1\}$: whether customer i received the coupon in the experiment (`test_coupon`)
- $Y_i$: observed revenue of customer i (revenue_after)
- $e=0.5$: probability of treatment (50/50 random assignment)


Intuitively, only customers whose experimental assignment matches our targeting decision are included.

Since these customers represent only half of their group (due to random assignment), their revenue is scaled up by $\frac 1e=2$ to estimate what the full group's revenue would be.

We apply this estimator to compare the following policies and select the one with the highest $\hat{R}(\pi)$: target nobody (baseline), target everyone, and our proposed targeting rule.

**Note**: `revenue_after` is recorded net of the 20% discount per the case documentation — no further cost adjustment is required.

In [ ]:
def evaluate_policy_ipw(df, targeting_col, outcome_col="revenue_after", p_treat=0.5):
    n = len(df)

    if targeting_col not in df.columns:
        raise ValueError(f"Column '{targeting_col}' not found in df.")
    if "test_coupon" not in df.columns:
        raise ValueError("Column 'test_coupon' not found in df.")
    if outcome_col not in df.columns:
        raise ValueError(f"Column '{outcome_col}' not found in df.")
    if not (0 < p_treat < 1):
        raise ValueError("p_treat must be in (0,1).")

    pi = df[targeting_col].astype(int)
    W  = df["test_coupon"].astype(int)
    Y  = df[outcome_col].astype(float)

    treated_match = (pi == 1) & (W == 1)
    control_match = (pi == 0) & (W == 0)


    ipw_total = (Y[treated_match] / p_treat).sum() + (Y[control_match] / (1 - p_treat)).sum()
    return ipw_total, ipw_total / n

In [ ]:
def compare_policies_ipw(data, policy_cols, outcome_col="revenue_after", p_treat=0.5, make_plot=True):
    """
    policy_cols: dict mapping {display_name: column_name_in_dat}
        {
          "No targeting": "target_none",
          "Target all": "target_all",
          "Option 2: OLS": "target_ols",
          "Option 1: Simple rule": "target_rule",
          "Option 3: Two-part": "target_twopart"
        }
    Returns:
      results_df with total and incremental revenue (vs No targeting)
    """
    dat = data.copy()

    if "No targeting" not in policy_cols:
        data["target_none"] = 0
        policy_cols = {"No targeting": "target_none", **policy_cols}

    baseline_col = policy_cols["No targeting"]
    if baseline_col not in data.columns:
        data[baseline_col] = 0

    # Evaluate totals
    rows = []
    for name, col in policy_cols.items():
        if col not in data.columns:
            continue

        total, per_user = evaluate_policy_ipw(data, col, outcome_col=outcome_col, p_treat=p_treat)
        rows.append({
            "policy": name,
            "policy_col": col,
            "targeted_n": int(data[col].sum()),
            "total_revenue_ipw": float(total),
            "revenue_per_user_ipw": float(per_user),
        })

    results = pd.DataFrame(rows).sort_values("total_revenue_ipw", ascending=False).reset_index(drop=True)

    # Compute incremental vs baseline
    baseline_total = results.loc[results["policy"] == "No targeting", "total_revenue_ipw"]
    if len(baseline_total) == 0:
        raise ValueError("Baseline 'No targeting' did not evaluate correctly.")
    baseline_total = float(baseline_total.iloc[0])

    results["incremental_vs_no_targeting"] = results["total_revenue_ipw"] - baseline_total

    return results


In [ ]:
def run_ipw_policy_comparison(data):
    data = data.copy()

    # Define baseline and target-all
    data["target_none"] = 0
    data["target_all"] = 1

    # Build policy map
    policy_cols = {
        "No targeting": "target_none",
        "Target all": "target_all",
    }
    if "target_ols" in data.columns:
        policy_cols["Option 2: OLS"] = "target_ols"
    if "target_rule" in data.columns:
        policy_cols["Option 1: Simple rule"] = "target_rule"
    if "target_twopart" in data.columns:
        policy_cols["Option 3: Two-part"] = "target_twopart"

    results = compare_policies_ipw(
        data,
        policy_cols=policy_cols,
        outcome_col="revenue_after",
        p_treat=0.5,
        make_plot=True
    )

    display(results)
    return results

In [ ]:
results = run_ipw_policy_comparison(data)

**Confidence Intervals via Bootstrap**

A single point estimate of $\hat{V}(\pi)$ is subject to sampling noise. To quantify uncertainty we use the **nonparametric bootstrap**:

1. Resample the dataset with replacement $B = 1{,}000$ times
2. Recompute $\hat{V}(\pi)$ on each resample $b$, yielding $\{\hat{V}^{(b)}\}_{b=1}^{B}$
3. Report the **2.5th and 97.5th percentiles** as the 95% confidence interval

$$\text{CI}_{95\%} = \left[\hat{V}^{(2.5\%)},\;\; \hat{V}^{(97.5\%)}\right]$$

A wide CI signals that the revenue estimate is sensitive to which customers happen to be in the sample — a reason to be cautious even if the point estimate looks good.

A narrow CI that lies clearly above the no-targeting baseline gives stronger evidence that the policy genuinely improves revenue.

In [ ]:
def bootstrap_ipw_revenue(data, target_col, outcome_col='revenue_after',
                          p_treat=None, n_bootstrap=1000, seed=42, return_raw=False):
    """
    Bootstrap IPW estimate of per-user policy value for a deterministic policy `target_col`.
    - data: DataFrame containing columns: outcome_col, 'test_coupon' (0/1), and target_col (0/1)
    - p_treat: if not None, use this as the randomization probability p; else use sample['test_coupon'].mean()
    - returns: (point_estimate, ci_low, ci_high)
    - set return_raw=True to also get raw bootstrap estimates array
    """
    rng = np.random.default_rng(seed)
    estimates = []

    # compute point estimate on the original sample
    def ipw_on_df(df):
        N = len(df)
        p = p_treat if p_treat is not None else df['test_coupon'].mean()
        # if p is exactly 0 or 1, estimator undefined
        if p == 0 or p == 1:
            return np.nan
        pi = df[target_col].astype(int)
        T = df['test_coupon'].astype(int)
        Y = df[outcome_col].astype(float)

        # contributions:
        mask_treated_and_policy1 = (pi == 1) & (T == 1)
        mask_control_and_policy0 = (pi == 0) & (T == 0)

        sum1 = (Y[mask_treated_and_policy1] / p).sum()
        sum0 = (Y[mask_control_and_policy0] / (1 - p)).sum()

        V_hat = (sum1 + sum0) / N
        return V_hat

    point_estimate = ipw_on_df(data)

    # bootstrap
    for _ in range(n_bootstrap):
        sample = data.sample(n=len(data), replace=True, random_state=int(rng.integers(0, 2**32 - 1)))
        est = ipw_on_df(sample)
        if np.isnan(est):
            # skip draws where p==0 or p==1 in that bootstrap sample
            continue
        estimates.append(est)

    if len(estimates) == 0:
        return point_estimate, np.nan, np.nan

    arr = np.array(estimates)
    ci_low, ci_high = np.percentile(arr, [2.5, 97.5])
    if return_raw:
        return point_estimate, ci_low, ci_high, arr
    return point_estimate, ci_low, ci_high

In [ ]:
from matplotlib.gridspec import GridSpecFromSubplotSpec

# Plot Forest plot
def plot_forest(ax, point_estimates, ci_results, policies):
    labels = list(point_estimates.keys())
    points = [point_estimates[l] for l in labels]
    ci_los = [ci_results[l][0] for l in labels]
    ci_his = [ci_results[l][1] for l in labels]

    for i, (label, point, lo, hi) in enumerate(zip(labels, points, ci_los, ci_his)):
        color = policies[label][1]
        ax.plot([lo, hi], [i, i], color=color, linewidth=2.5, zorder=2)
        ax.plot([lo, lo], [i - 0.15, i + 0.15], color=color, linewidth=2.5)
        ax.plot([hi, hi], [i - 0.15, i + 0.15], color=color, linewidth=2.5)
        ax.scatter(point, i, color=color, s=120, zorder=3)
        ax.text(point, i + 0.25, f'${point:.3f}',
                ha='center', va='bottom',
                color=color, fontweight='bold', fontsize=9)

    ax.axvline(x=point_estimates['No targeting'], color='gray', linestyle='--',
               linewidth=1.2, alpha=0.6, label='No targeting baseline')
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel('IPW Revenue per User ($)', fontsize=10)
    ax.set_title('Policy Comparison\nPoint estimate + 95% CI', fontweight='bold', fontsize=10)
    ax.legend(fontsize=8, loc='lower left')
    ax.spines[['top', 'right']].set_visible(False)


# Plot IPW bar chart
def plot_ipw_bars(ax, results_df):
    plot_df = results_df.sort_values("incremental_vs_no_targeting", ascending=False)
    vals  = plot_df["incremental_vs_no_targeting"].values
    names = plot_df["policy"].values
    names = [n.replace(': ', ':\n') for n in plot_df["policy"].values]
    colors = ["green" if v >= 0 else "red" for v in vals]

    bars = ax.bar(names, vals, color=colors, edgecolor="white", width=0.6)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_ylabel("Incremental Revenue ($) vs No Targeting", fontsize=10)
    ax.set_title("IPW Policy Comparison\nIncremental Total Revenue", fontweight='bold', fontsize=10)
    ax.tick_params(axis='x', labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)

    for bar, v in zip(bars, vals):
        x = bar.get_x() + bar.get_width() / 2
        offset = +6 if v >= 0 else -6
        va     = "bottom" if v >= 0 else "top"
        ax.annotate(f"${v:,.0f}",
                    xy=(x, v),
                    xytext=(0, offset),
                    textcoords="offset points",
                    ha="center", va=va,
                    fontsize=8, fontweight="bold")


# Plot Agreement heatmaps
def plot_targeting_agreement(fig, gs_spec, data):
    cols = [c for c in ['target_ols', 'target_rule', 'target_twopart'] if c in data.columns]
    if len(cols) < 2:
        return

    label_map = {'target_ols': 'OLS', 'target_rule': 'Simple Rule', 'target_twopart': 'Two-Part'}
    pairs = [(cols[i], cols[j]) for i in range(len(cols)) for j in range(i + 1, len(cols))]
    n = len(pairs)
    pct_agree = (data[cols].nunique(axis=1) == 1).mean() * 100

    inner = GridSpecFromSubplotSpec(1, n + 1, subplot_spec=gs_spec,
                                    width_ratios=[1] * n + [0.6], wspace=0.35)
    axes = [fig.add_subplot(inner[0, i]) for i in range(n + 1)]

    for ax, (c1, c2) in zip(axes[:n], pairs):
        ct = pd.crosstab(data[c1], data[c2])
        ct_pct = ct / ct.values.sum() * 100

        ax.imshow(ct_pct.values, cmap='Blues', vmin=0, vmax=100, aspect='auto')
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(['No', 'Target'], fontsize=8)
        ax.set_yticklabels(['No', 'Target'], fontsize=8)
        ax.set_xlabel(label_map[c2], fontsize=9)
        ax.set_ylabel(label_map[c1], fontsize=9)
        ax.set_title(f'{label_map[c1]} vs {label_map[c2]}', fontsize=9, fontweight='bold')

        for i in range(2):
            for j in range(2):
                v = ct_pct.values[i, j]
                ax.text(j, i, f'{ct.values[i,j]:,}\n({v:.1f}%)',
                        ha='center', va='center', fontsize=8,
                        color='white' if v > 50 else 'black', fontweight='bold')
                ax.grid(False)

    # Summary panel
    ax_s = axes[-1]
    ax_s.axis('off')
    ax_s.text(0.5, 0.80, "Agreement", ha='center', fontsize=9, color='gray',
              transform=ax_s.transAxes)
    ax_s.text(0.5, 0.62, f"{pct_agree:.1f}%", ha='center', fontsize=26,
              fontweight='bold', color='blue', transform=ax_s.transAxes)
    for i, c in enumerate(cols):
        ax_s.text(0.5, 0.42 - i * 0.14,
                  f"{label_map[c]}: {data[c].sum():,} targeted",
                  ha='center', fontsize=8, color='#333', transform=ax_s.transAxes)

Option 1 (Simple Rule) generates the highest estimated total revenue, followed closely by Option 2 (OLS). All three targeting strategies outperform both the no-targeting baseline and the target-all approach, confirming that selective targeting improves revenue.

In [ ]:
data_plot = data.copy()
data_plot['target_none'] = 0
data_plot['target_all']  = 1

policies = {
    'No targeting'         : ['target_none', 'gray'],
    'Target all'           : ['target_all', 'red'],
    'Option 3: Two-part'   : ['target_twopart', 'mediumpurple'],
    'Option 2: OLS'        : ['target_ols', 'orange'],
    'Option 1: Simple rule': ['target_rule', 'lightseagreen'],
}

# Point estimates (from IPW results)
point_estimates = dict(zip(results['policy'], results['revenue_per_user_ipw']))

print("Bootstrapping CIs ...")
ci_results = {}
for label, col in policies.items():
    if col[0] in data_plot.columns:
        point_estimate, ci_lo, ci_hi = bootstrap_ipw_revenue(data_plot, col[0])
        ci_results[label] = (ci_lo, ci_hi)
        print(f"  {label}: [{ci_lo:.3f}, {ci_hi:.3f}]")

In [ ]:
# Compose the full figure
fig = plt.figure(figsize=(16, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 0.85], hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

plot_ipw_bars(ax1, results)
plot_forest(ax2, point_estimates, ci_results, policies)

plot_targeting_agreement(fig, gs[1, :], data)

fig.suptitle("Targeting Strategy — Evaluation", fontweight='bold')
plt.show()

**Comparison of Incremental Values:** The comparison table and bar chart show incremental revenue relative to the no-targeting baseline for each policy. IPW ensures these estimates are unbiased despite the experimental assignment not matching our targeting rules.

**Confidence Intervals** Bootstrapped 95% confidence intervals quantify the uncertainty around each policy's IPW revenue-per-user estimate, supporting a statistically grounded comparison across options.

**Agreement matrix:** The agreement matrix highlights customers where policies diverge — these marginal cases are most sensitive to model choice and warrant closer inspection.

All three strategies agree on 47.3% of users — but the disagreement is entirely one-directional. Every user targeted by Option 1 is also targeted by both the OLS and Two-Part models (100% inclusion), meaning no model-based approach rejects any of the simple rule's targets. The lower overall agreement is driven solely by the models targeting additional marginal users (2,089 extra for OLS, 2,634 for Two-Part) that the rule excludes. However, the IPW evaluation shows that including these extra users does not improve total revenue — it slightly reduces it.

### Pros and Cons

**Option 1 — Simple Rule**

*Pros*

- In our view, Option 1 delivers the strongest empirical performance, generating the highest IPW-adjusted total revenue (42,539 USD) and the largest incremental lift relative to no targeting and the other two options.

- It targets only 1,030 customers, which makes it efficient and avoids unnecessary coupon distribution.

- The rule is transparent and easy to interpret, since it is directly grounded in observed treatment heterogeneity.

- Operationally, it is simple to implement and does not require complex model maintenance.

*Cons*

- We recognize that the rule is relatively rigid and threshold-based, which limits flexibility.

- It may not generalize well if customer behavior changes over time.

- Because it is not based on a full predictive model, it may miss more subtle patterns of heterogeneity.

- It does not easily adapt to different budget constraints or alternative targeting proportions.

**Option 2 — OLS-Based Targeting**

*Pros*

- In our assessment, Option 2 provides strong performance (42,206 USD total revenue and +3,335.92 USD incremental lift), very close to Option 1.

- It incorporates behavioral and demographic factors simultaneously, allowing for more systematic and data-driven targeting.

- The model supports ranking customers by predicted uplift, making it flexible and scalable.

- It can be retrained and updated easily as new data becomes available.

*Cons*

- It relies on linear functional form assumptions.

- It treats revenue as a single continuous outcome and does not explicitly model the two-stage purchase process.

- Despite its flexibility, it does not achieve the highest total revenue in our evaluation.

**Option 3 — Two-Part Model**

*Pros*

- We find this approach methodologically appealing because it separates conversion from spending.

- It better reflects the economic structure of revenue generation, especially when many users have zero revenue.

- It provides deeper insight into whether the coupon affects purchase probability or spending intensity.

*Cons*

- In our evaluation, it generates lower total revenue (41,938 USD) and lower incremental lift (+3,068.76 USD) compared to Options 1 and 2.

- The added modeling complexity does not translate into superior policy performance in this case.

- It is more complex to implement and maintain operationally.

# 5. Conclusion

In this project, we examined not only whether the coupon works on average, but also how it should be implemented. First, we verified that the A/B design supports causal interpretation. The treatment and control groups were balanced, and treatment assignment was not systematically related to pre-treatment characteristics, supporting the validity of the experiment.

On average, the coupon increases the number of transactions but does not generate a statistically significant increase in revenue. This suggests that sending coupons to all users would likely reduce margins without meaningfully increasing total revenue.

We therefore focused on heterogeneity. Two variables are especially important: shopping_cart and num_past_purch. Users who added items to their cart respond more strongly to the coupon, while the effect decreases as past purchase frequency increases.

We evaluated three targeting strategies:

**Option 1** – Simple rule-based targeting: target users who added items to the cart and have a low number of past purchases (e.g., ≤ 2). This rule is transparent and easy to implement, and it is directly based on the strongest interaction patterns observed in the data.

**Option 2** – OLS uplift model: estimate individual-level treatment effects using a regression model with interaction terms and target users with positive predicted uplift.

**Option 3** – Two-part model: separately model purchase probability and spending amount to better account for zero revenue outcomes, and compute uplift based on both components.

**Option 1 delivers the highest estimated revenue improvement among all evaluated strategies, and for this reason we selected it as our final policy.**

Although we initially expected to favor a more advanced model, the results clearly show that the simple rule-based approach performs best under our revenue-focused comparison. In this analysis, we evaluated strategies purely based on revenue outcomes and did not incorporate implementation or model complexity costs. Within this framework, Option 1 provides the strongest performance. While more complex approaches such as Option 2 or Option 3 could become preferable if additional operational or modeling considerations were included, they do not outperform Option 1 in our current setting.


**Final recommendation**: adopt the cart-intent, low-history targeting policy (Option 1) as the primary rule for the next campaign, and implement it as the submission-ready id / target file. This policy is grounded in both the heterogeneity evidence and the policy evaluation results, it avoids over-distributing coupons, and it directly aligns with the core economic insight from the experiment: coupons create incremental revenue mainly when they convert high-intent users who are not already frequent buyers.

**Limitations and next steps**: our evaluation relies on one experimental sample and a limited set of covariates; performance could shift if user mix or channels change. In a real deployment, we would re-estimate thresholds periodically, monitor drift, and run a follow-up test where the targeting rule itself is randomized (policy A/B) to confirm out-of-sample performance.

# Next Campaign Targeting

We apply our recommended targeting rule (Option 1: `num_past_purch ≤ 2` AND `shopping_cart = 1`) to the 6,000 next-campaign users.

In [ ]:
data_next = pd.read_excel('IMA2026_WebsiteCaseData_with_Demog.xlsx',
                         sheet_name='Next_Campaign_demog')

data_next['id'] = data_next['id'].astype(str)

data_next['target'] = (
    (data_next['num_past_purch'] <= 2) & (data_next['shopping_cart'] == 1)
).astype(int)

print(f"Next campaign pool: {len(data_next)} users")
print(f"Targeted:           {data_next['target'].sum()} ({data_next['target'].mean():.1%})")
print(f"Not targeted:       {(data_next['target'] == 0).sum()}")

submission = data_next[['id', 'target']]
submission.head()

To cross-validate, we also score all users using the OLS and Two-Part models fitted on the AB test data, creating counterfactual predictions under coupon vs. no coupon for each user. We then project total revenue for each strategy: Option 1 uses the experimental ATE from the target segment, while Options 2 and 3 use the sum of their model-predicted uplifts for targeted users.

In [ ]:
data_next['channel_acq'] = data_next['channel_acq'].replace({
    1: "Google", 2: "Facebook", 3: "Instagram", 4: "Referral", 5: "Other"
})
baseline_per_user = data.loc[data['test_coupon'] == 0, 'revenue_after'].mean()
n_total = len(data_next)

# Coupon to no one
rev_no_coupon = baseline_per_user * n_total

# Option 1: experimental ATE × targeted users
rev_option1 = rev_no_coupon + (ATE * data_next['target'].sum())

next_t1 = data_next.copy()
next_t1['test_coupon'] = 1

next_t0 = data_next.copy()
next_t0['test_coupon'] = 0

# Option 2: baseline + sum of OLS uplift for targeted users

data_next['uplift_ols'] = model_ols.predict(next_t1) - model_ols.predict(next_t0)
data_next['target_ols'] = (data_next['uplift_ols'] > 0).astype(int)
rev_option2 = rev_no_coupon + data_next.loc[data_next['target_ols'] == 1, 'uplift_ols'].sum()

# Option 3: baseline + sum of Two-Part uplift for targeted users
pA_t1  = model_partA.predict(next_t1)
pA_t0  = model_partA.predict(next_t0)
rev_t1 = model_partB.predict(next_t1)
rev_t0 = model_partB.predict(next_t0)

data_next['uplift_twopart'] = pA_t1 * rev_t1 - pA_t0 * rev_t0
data_next['target_twopart'] = (data_next['uplift_twopart'] > 0).astype(int)
rev_option3 = rev_no_coupon + data_next.loc[data_next['target_twopart'] == 1, 'uplift_twopart'].sum()

# Coupon to everyone
treat_per_user = data.loc[data['test_coupon'] == 1, 'revenue_after'].mean()
rev_all = treat_per_user * n_total

projection = pd.DataFrame({
    'Strategy': ['No coupons', 'Coupon to everyone',
                 'Option 1: Simple Rule', 'Option 2: OLS', 'Option 3: Two-Part'],
    'Total Revenue ($)': [rev_no_coupon, rev_all, rev_option1, rev_option2, rev_option3],
    'Incremental ($)': [0, rev_all - rev_no_coupon,
                        rev_option1 - rev_no_coupon,
                        rev_option2 - rev_no_coupon,
                        rev_option3 - rev_no_coupon],
    'Users Targeted': [0, 6000,
                       data_next['target'].sum(),
                       data_next['target_ols'].sum(),
                       data_next['target_twopart'].sum()],
    '% of Pool': [0, 100,
                  data_next['target'].mean() * 100,
                  data_next['target_ols'].mean() * 100,
                  data_next['target_twopart'].mean() * 100]
})

projection['Total Revenue ($)'] = projection['Total Revenue ($)'].apply(lambda x: f"${x:,.0f}")
projection['Incremental ($)'] = projection['Incremental ($)'].apply(lambda x: f"${x:+,.0f}" if x != 0 else '(baseline)')
projection['Users Targeted'] = projection['Users Targeted'].apply(lambda x: f"{x:,}" if x > 0 else '—')
projection['% of Pool'] = projection['% of Pool'].apply(lambda x: f"{x:.1f}%" if x > 0 else '—')

projection

All three targeting strategies project positive incremental revenue over the no-coupon baseline, while couponing everyone is projected to reduce revenue. Option 2 (OLS) shows a slightly higher projection, though this reflects model-predicted uplift across a much larger target set. Option 1 achieves a comparable increment while targeting only about 20% of users, making it the most efficient strategy in terms of revenue per coupon sent

In [ ]:
# Export submission file

submission.to_csv('targeting_decision.csv', index=False)
print(f"Saved: targeting_decision.csv ({len(submission)} rows)")

 We submit Option 1 as our final targeting decision.